# Exercise 04: Attention


# 🌍 Neural Machine Translation with Attention (English → German)

### 🎯 Learning Objectives
By the end of this lab, you should be able to:

- Convert sentences into numerical representations using word embeddings.
- Build a **Seq2Seq model** with **LSTM encoder–decoder**.
- Understand and implement different **attention mechanisms**.
- Visualize **attention weights** and interpret them as alignment heatmaps.

---

### 🧩 Tasks Overview
1. **Data Preparation:** Load and preprocess the English–German dataset.  
2. **Baseline Model:** Train a simple LSTM encoder–decoder (fixed context).  
3. **Attention Models:** Extend the model with attention mechanisms.  
4. **Visualization:** Generate translations and visualize attention weights.  
5. **Discussion:** Reflect on what attention helps the model achieve.  

> 💡 As you go through each section, think about:  
> - What problem does each new mechanism solve?  
> - How does attention change the model’s behavior?

The default `FAST_MODE` is intended for a tutorial. Set it to `False` for a longer experiment with the original sample size and training schedule.


In [ ]:
# Run once if the JupyterHub image does not already provide these packages.
# `%pip` installs into the currently selected notebook kernel.
%pip install -q torch pandas numpy matplotlib seaborn transformers sentencepiece


If package installation is disabled on the JupyterHub, use a user environment (or ask the administrator for the course kernel):

1. `python -m venv myenv`
2. `source myenv/bin/activate`
3. `python -m pip install --upgrade pip setuptools wheel`
4. `python -m pip install ipykernel torch pandas numpy matplotlib seaborn transformers sentencepiece`
5. `python -m ipykernel install --user --name myenv --display-name "Python (myenv)"`
6. Select **Python (myenv)** as the notebook kernel and restart it.

The final T5 section also needs internet access the first time the public model is downloaded. No Hugging Face login is required for that public checkpoint.


## 🔧 Setup & Package Installation

This section:
- Imports the required libraries.
- Sets the random seed for reproducibility.
- Detects GPU/CPU device.

In [ ]:
# Core Python utilities
import math
import random
import re
import time
from collections import Counter
from pathlib import Path

# Numerical and data handling
import numpy as np
import pandas as pd

# Deep learning with PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

random_state = 10
random.seed(random_state)
np.random.seed(random_state)
torch.manual_seed(random_state)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(random_state)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


## 🔤 Data Preparation & Preprocessing

**Goal:** convert raw text pairs (English → German) into tensors that can be fed to the model.

Typical steps:
1. **Tokenize** source and target sentences.
2. **Build vocabularies** (or use pretrained subword models, e.g., SentencePiece/BPE).
3. **Add special tokens**: `<sos>`, `<eos>`, `<pad>`.
4. **Pad/trim** to a max length.

> ❓ *Think about it:*  
> - How does maximum sequence length affect memory usage and translation quality?  
> - Why do we need `<sos>` and `<eos>` in the target?

In [ ]:
# Dataset and language columns

# If you use Google Colab, you can uncomment the following lines
# from google.colab import drive
# drive.mount('/content/gdrive')
# DATASET = Path("/content/gdrive/MyDrive/Colab Notebooks/eng-deu.tsv")

DATASET = Path("eng-deu.tsv")
SOURCE_LANGUAGE = "English"
TARGET_LANGUAGE = "German"

# Tutorial-sized defaults; use the larger values for an after-class run.
FAST_MODE = True
SAMPLE_SIZE = 5_000 if FAST_MODE else 20_000

if not DATASET.exists():
    raise FileNotFoundError(f"Place {DATASET.name!r} in the same folder as this notebook.")

# `usecols` deliberately ignores the legacy row-number column in the TSV file.
data = pd.read_csv(
    DATASET,
    sep="\t",
    usecols=[SOURCE_LANGUAGE, TARGET_LANGUAGE],
).dropna().drop_duplicates()

n = min(SAMPLE_SIZE, len(data))
data = data.sample(n=n, random_state=random_state).reset_index(drop=True)

# Split raw rows before building the vocabulary, so validation text cannot leak
# token-frequency information into training.
split = int(0.9 * len(data))
train_data = data.iloc[:split].reset_index(drop=True)
val_data = data.iloc[split:].reset_index(drop=True)
print(f"Rows: {len(data):,} (train={len(train_data):,}, validation={len(val_data):,})")
data.head()


In [ ]:
# Lowercase text, separate sentence punctuation, and remove other characters.
def normalize(s):
    s = str(s).lower().strip()
    s = re.sub(r"([?.!])", r" \1 ", s)
    s = re.sub(r"[^a-zäöüß?.!]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def tokenize(sentence):
    return normalize(sentence).split()


# TODO1: Fill in the blanks in the vocabulary construction.
def build_vocab(sentences, min_freq=2):
    counter = Counter()
    for sentence in sentences:
        counter.update(tokenize(_____))

    specials = ["<pad>", "<sos>", "<eos>", "<unk>"]
    # Sorting makes the token IDs deterministic when frequencies tie.
    words = sorted(token for token, count in counter.items() if count >= _____)
    vocab = specials + words
    stoi = {token: _____ for index, token in enumerate(vocab)}
    itos = {index: _____ for token, index in stoi.items()}
    return vocab, stoi, itos


# Build vocabularies from training data only.
src_vocab, src_stoi, src_itos = build_vocab(train_data[SOURCE_LANGUAGE])
tgt_vocab, tgt_stoi, tgt_itos = build_vocab(train_data[TARGET_LANGUAGE])
print("Source vocabulary:", len(src_vocab))
print("Target vocabulary:", len(tgt_vocab))

PAD_SRC = src_stoi["<pad>"]
PAD_TGT = tgt_stoi["<pad>"]


In [ ]:
MAX_LEN = 15
BATCH_SIZE = 64


# TODO2: Fill in the blanks for truncation, unknown tokens, and padding.
def tensorize(sentence, stoi, max_len=MAX_LEN):
    tokens = ["<sos>"] + tokenize(sentence)[: _____] + ["<eos>"]
    ids = [stoi.get(token, stoi[_____]) for token in tokens]
    ids += [stoi["<pad>"]] * (_____ - len(ids))
    return torch.tensor(ids, dtype=torch.long)


def tensorize_pairs(frame):
    return [
        (tensorize(src, src_stoi), tensorize(tgt, tgt_stoi))
        for src, tgt in zip(frame[SOURCE_LANGUAGE], frame[TARGET_LANGUAGE])
    ]


train_pairs = tensorize_pairs(train_data)
val_pairs = tensorize_pairs(val_data)


def collate(batch):
    srcs, tgts = zip(*batch)
    return torch.stack(srcs).to(DEVICE), torch.stack(tgts).to(DEVICE)


train_loader = DataLoader(
    train_pairs, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate
)
val_loader = DataLoader(
    val_pairs, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate
)
print("Loaded dataset. Train pairs:", len(train_pairs), "- Validation pairs:", len(val_pairs))


### ⚙️ Hyperparameters

This cell initializes the environment and sets key training parameters.

We begin with **lightweight defaults** so training runs quickly in class.

<div align="left">
    
| Category | Hyperparameter | Purpose |
|:-----------|:----------------|:----------|
| Model | `EMB_SIZE` | Word-embedding dimension. |
| | `HID_SIZE` | Hidden-state size of LSTMs. |
| | `DROPOUT` | Regularization; prevents overfitting. |
| Optimization | `LEARNING_RATE` | Step size for parameter updates. |
| | `GRAD_CLIP` | Limit gradient norm to avoid explosions. |
| Training | `N_EPOCHS` | Number of passes over the data. |
| | `BATCH_SIZE` | Mini-batch size; affects stability & speed. |
| | `TEACHER_FORCING` | Probability of using the *true* previous token (`<sos>`-based) instead of the model’s prediction during training. |
| Misc | `MAX_LEN` | Maximum sentence length (padding longer ones). |

</div>

---

### 🧠 Think About
- What happens if you **reduce** `TEACHER_FORCING` to 0?  
- How does increasing `HID_SIZE` influence translation quality and training time?  
- Why is gradient clipping important for LSTMs?

In [ ]:
class Config:
    emb = 128 if FAST_MODE else 256
    hid = 256 if FAST_MODE else 512
    dropout = 0.25
    lr = 1e-3
    epochs = 4 if FAST_MODE else 20
    batch_size = BATCH_SIZE
    teacher_forcing = 0.7
    max_len = MAX_LEN
    grad_clip = 1.0


CFG = Config()
print("Number of batches per epoch:", len(train_loader))


## 🔄 Sequence-to-Sequence (Seq2Seq) Models

A **Sequence-to-Sequence (Seq2Seq)** model is designed to transform one sequence into another — for example, translating an English sentence into German, summarizing a paragraph, or converting speech to text.  
Unlike traditional models that output a single label, Seq2Seq models handle **variable-length inputs and outputs**.

### 🧩 Core Idea

A Seq2Seq model has two main components:

1. **Encoder** – reads and processes the input sequence (e.g., an English sentence) one token at a time.  
   As it reads, it updates its hidden state to capture the meaning of what it has seen so far.  
   After the last token, the encoder produces a **context vector**, which acts as a summary of the entire input.

2. **Decoder** – generates the output sequence (e.g., the translated German sentence) using the context vector as its starting point.  
   It predicts one token at a time, using its own previous outputs to guide the next prediction.

In [ ]:
# Encoder: reads variable-length input sequences and produces hidden states.
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb, hid, dropout, bidirectional=False):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=PAD_SRC)
        self.drop = nn.Dropout(dropout)
        self.bidirectional = bidirectional
        self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=bidirectional)

        if bidirectional:
            self.fc_h = nn.Linear(2 * hid, hid)
            self.fc_c = nn.Linear(2 * hid, hid)

    def forward(self, src):
        embedded = self.drop(self.emb(src))
        lengths = (src != PAD_SRC).sum(dim=1).cpu()
        packed = pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False
        )
        packed_out, (h, c) = self.lstm(packed)
        out, _ = pad_packed_sequence(
            packed_out, batch_first=True, total_length=src.size(1)
        )

        if self.bidirectional:
            h = self.fc_h(torch.cat((h[-2], h[-1]), dim=1)).unsqueeze(0)
            c = self.fc_c(torch.cat((c[-2], c[-1]), dim=1)).unsqueeze(0)
        return out, (h, c)


class DecoderNoAttn(nn.Module):
    def __init__(self, vocab_size, emb, hid, dropout):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=PAD_TGT)
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(emb, hid, batch_first=True)
        self.fc = nn.Linear(hid, vocab_size)

    def forward(self, tgt_step, hidden):
        # TODO3: Fill in the input, recurrent state, and projected output.
        embedded = self.drop(self.emb(_____))
        out, hidden = self.lstm(embedded, _____)
        return self.fc(_____), hidden


class Seq2SeqNoAttn(nn.Module):
    def __init__(self, emb, hid, dropout):
        super().__init__()
        self.enc = Encoder(len(src_vocab), emb, hid, dropout)
        self.dec = DecoderNoAttn(len(tgt_vocab), emb, hid, dropout)


### 🧠 The Bottleneck Problem

Because the encoder compresses the entire input into a **single fixed-size vector**, it may struggle to remember all the important details, especially for **long or complex sentences**.  
This limitation makes it hard for the decoder to reproduce accurate or fluent outputs when the input is lengthy.

> 💭 **Think about it:**  
> - What challenges might arise when trying to summarize a long sentence with just one vector?  
> - How might we help the model focus on the most important parts of the input instead of treating all words equally?

## 🧭 Introducing Attention Mechanisms

To solve the bottleneck problem, we use **attention mechanisms**.  
Instead of relying on a single vector, the decoder learns to **attend** (or focus) on different parts of the input sequence at each step.

We’ll start with the simplest version: **Luong (dot-product) attention**, then move to **multi-head attention** (used in Transformers), and finally look at a **self-attention encoder**.

In [ ]:
# Luong Attention (dot-product attention)
class LuongAttention(nn.Module):
    def __init__(self, hid):
        super().__init__()
        # Learnable linear transformation for the query vector
        self.W = nn.Linear(hid, hid, bias=False)

    def forward(self, query, keys, values, key_mask=None):
        # TODO4: Fill in the blanks in masked Luong attention.
        # Compute attention scores: similarity between query and each encoder output
        scores = torch.bmm(self.W(_____), keys.transpose(1, 2))  # (B, 1, T)

        # Apply mask to ignore padding tokens
        if key_mask is not None:
            mask_val = torch.finfo(scores.dtype).min  # Smallest representable value for current dtype
            scores = scores.masked_fill(~key_mask.unsqueeze(1), _____)

        # Normalize scores to probabilities
        attn = F.softmax(scores, dim=_____)

        # Compute context vector as a weighted sum of encoder outputs
        ctx = torch.bmm(attn, _____)

        return ctx, attn

### 🧩 Decoder with Luong Attention

We now use the attention mechanism inside the **decoder**.  
At each time step:
1. The decoder generates a **query** from its hidden state.
2. The attention layer compares this query with all encoder outputs.
3. It returns a **context vector** that emphasizes relevant words.
4. The decoder combines this context with the current word embedding to produce the next output.

In [ ]:
# Decoder with Luong "general" attention.
class DecoderLuong(nn.Module):
    def __init__(self, vocab_size, emb, hid, dropout, enc_dim=None):
        super().__init__()
        enc_dim = hid if enc_dim is None else enc_dim
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=PAD_TGT)
        self.drop = nn.Dropout(dropout)
        self.enc_proj = nn.Identity() if enc_dim == hid else nn.Linear(enc_dim, hid)
        self.lstm = nn.LSTM(emb + hid, hid, batch_first=True)
        self.fc = nn.Linear(hid, vocab_size)
        self.attn = LuongAttention(hid)

    def forward(self, tgt_step, enc_out, hidden, key_mask=None):
        embedded = self.drop(self.emb(tgt_step))
        query = hidden[0][-1].unsqueeze(1)
        enc_out = self.enc_proj(enc_out)
        context, weights = self.attn(query, enc_out, enc_out, key_mask)
        out, hidden = self.lstm(torch.cat([embedded, context], dim=-1), hidden)
        return self.fc(out), hidden, weights


class Seq2SeqLuong(nn.Module):
    def __init__(self, emb, hid, dropout):
        super().__init__()
        self.enc = Encoder(len(src_vocab), emb, hid, dropout)
        self.dec = DecoderLuong(len(tgt_vocab), emb, hid, dropout)


## ⚖️ From Luong to Scaled Dot-Product and Multi-Head Attention

In **Luong attention**, we compute a dot product between the decoder’s hidden state (the **query**) and each encoder hidden state (the **key**).  
This tells the model how much to attend to each part of the input sequence.

While effective, this approach has a few **shortcomings**:

- 🔸 The dot product values grow with the hidden size, producing **large attention scores**.  
  When passed through a softmax, this leads to **extremely peaked distributions** and **unstable gradients**.  
- 🔸 In this notebook, the **recurrent decoder** produces one query at a time. The attention formula itself can process many queries in parallel, as Transformers do.  
- 🔸 There is only **a single attention head**, so the model can capture only one type of relationship at each step.

To address these issues, the **Transformer architecture** introduced a refined version of dot-product attention — the **Scaled Dot-Product Attention** — and extended it to multiple **parallel heads** for richer representation learning.


---

### 🔢 Query, Key, and Value (Q, K, V)

All modern attention mechanisms, including Luong and Transformer attention, are built around three core components:

- **Query (Q):** What we are trying to find information for (e.g., the decoder’s current hidden state).  
- **Key (K):** What each source position offers — used for comparison with the query.  
- **Value (V):** The actual information retrieved from the encoder outputs.

The attention mechanism compares the **query** with all **keys** (e.g., via dot products), applies a softmax to obtain weights, and then takes a weighted average of the **values**.

---

### 🧮 Scaled Dot-Product Attention

To stabilize the computation, we divide the raw dot products by the square root of the key dimension \( d_k \):

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V
$$

This **scaling factor** keeps the attention scores in a reasonable range, preventing the softmax from becoming too peaked when \( d_k \) is large and making training more stable.

---

In [ ]:
# Scaled Dot-Product Attention
class ScaledDotProduct(nn.Module):
    def forward(self, Q, K, V, key_mask=None):
        # TODO5: Fill in the scaling dimension, mask, and value tensor.
        d = Q.size(_____)
        # Compute scaled dot-product scores
        scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(_____)
        if key_mask is not None:
            mask_val = torch.finfo(scores.dtype).min
            scores = scores.masked_fill(~key_mask.unsqueeze(1), _____)
        attn = F.softmax(scores, dim=_____)
        ctx = torch.bmm(attn, _____)
        return ctx, attn

### 🧠 Multi-Head Attention

Rather than relying on a single attention operation, **multi-head attention** performs several scaled dot-product attentions in parallel.  
Each head has its own learned projections of \( Q, K, V \), allowing it to capture **different types of relationships**:

- One head might focus on **syntactic** structure.  
- Another might capture **semantic** similarity.  
- Others can model **long-range dependencies**.

Finally, the outputs of all heads are concatenated and projected to form a unified, context-rich representation.

---

In [ ]:
# Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, hid, heads):
        super().__init__()
        assert hid % heads == 0
        self.h, self.dh = heads, hid // heads
        self.Wq = nn.Linear(hid, hid)
        self.Wk = nn.Linear(hid, hid)
        self.Wv = nn.Linear(hid, hid)
        self.out = nn.Linear(hid, hid)
        self.dot = ScaledDotProduct()

    # Split hidden states into multiple heads
    # TODO6: Fill in the missing head dimensions and tensors.
    def _split(self, x):  # (B, T, H) → (B*h, T, dh)
        B, T, H = x.size()
        return x.view(B, T, self.h, self.dh).transpose(1, 2).reshape(B * _____, T, _____)

    # Merge heads back into one tensor
    def _merge(self, x, B, T):  # (B*h, T, dh) → (B, T, H)
        return x.view(B, self.h, T, self.dh).transpose(1, 2).reshape(B, T, _____)

    def forward(self, Q, K, V, key_mask=None):
        B, Tq, H = Q.size()
        Tk = K.size(1)
        Qh, Kh, Vh = self._split(self.Wq(Q)), self._split(self.Wk(K)), self._split(self.Wv(V))
        if key_mask is not None:
            key_mask = key_mask.repeat_interleave(_____, dim=0)
        ctx, attn = self.dot(Qh, Kh, Vh, key_mask=key_mask)
        ctx = self._merge(ctx, B, Tq)
        return self.out(_____), attn


### 🧩 Decoder with Multi-Head Attention

Here we combine the LSTM decoder with **multi-head attention**.  
Each attention head focuses on different relationships between input and output tokens,  
and their combined output helps the model capture richer contextual information.

In [ ]:
heads = 4

# Decoder with Multi-Head Attention
class DecoderMH(nn.Module):
    def __init__(self, vocab_size, emb, hid, heads, dropout):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(emb, hid, batch_first=True)
        self.mh = MultiHeadAttention(hid, heads=heads)
        self.fc = nn.Linear(hid, vocab_size)

    def forward(self, tgt_step, enc_out, hidden, key_mask=None):
        emb = self.drop(self.emb(tgt_step))         # (B, 1, E)
        out, hidden = self.lstm(emb, hidden)        # (B, 1, H)
        ctx, attn = self.mh(out, enc_out, enc_out, key_mask=key_mask)
        logits = self.fc(out + ctx)                 # Residual connection
        return logits, hidden, attn


# Full Seq2Seq Model with Multi-Head Attention
class Seq2SeqMH(nn.Module):
    def __init__(self, emb, hid, heads, dropout):
        super().__init__()
        self.enc = Encoder(len(src_vocab), emb, hid, dropout)
        self.dec = DecoderMH(len(tgt_vocab), emb, hid, heads, dropout)


## 🧱 Self-Attention Encoder

The previous attention modules are **cross-attention**: a decoder query retrieves information from encoder keys and values. Here the encoder itself performs **self-attention**, so every source token can directly use every other non-padding source token.

Self-attention alone does not encode token order. We therefore add learned positional embeddings before attention. This section replaces only the recurrent **encoder**; the decoder remains an LSTM so that we can isolate the encoder change.


In [ ]:
class SelfAttnEncoder(nn.Module):
    def __init__(self, vocab_size, emb, hid, heads, dropout, max_len=MAX_LEN):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=PAD_SRC)
        self.pos = nn.Embedding(max_len, hid)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Linear(emb, hid)
        self.mha = nn.MultiheadAttention(
            hid, heads, dropout=dropout, batch_first=True
        )
        self.ff = nn.Sequential(
            nn.Linear(hid, 4 * hid), nn.ReLU(), nn.Linear(4 * hid, hid)
        )
        self.ln1 = nn.LayerNorm(hid)
        self.ln2 = nn.LayerNorm(hid)

    def forward(self, src):
        # TODO7: Fill in positions, padding, Q/K/V, and the valid-token mask.
        positions = torch.arange(_____, device=src.device).unsqueeze(0)
        x = self.proj(self.drop(self.emb(src))) + self.pos(positions)
        padding_mask = src.eq(_____)

        attn_out, _ = self.mha(x, _____, _____, key_padding_mask=padding_mask)
        x = self.ln1(x + self.drop(attn_out))
        x = self.ln2(x + self.drop(self.ff(x)))

        # Masked mean: padding queries must not influence the decoder state.
        valid = (_____).unsqueeze(-1)
        pooled = (x * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1)
        h_last = pooled.unsqueeze(0)
        return x, (h_last, h_last.clone())


class Seq2SeqSelfEnc(nn.Module):
    def __init__(self, emb, hid, heads, dropout):
        super().__init__()
        self.enc = SelfAttnEncoder(len(src_vocab), emb, hid, heads, dropout)
        self.dec = DecoderMH(len(tgt_vocab), emb, hid, heads, dropout)


## ⚙️ Training the Seq2Seq Model

Once we define our encoder and decoder, we need to **train** the model so that it learns how to translate from the source language (English) to the target language (German).

---

### 🧠 Training Overview

Each training step follows this process:

1. **Encoder** reads the entire source sentence and produces a set of hidden states that represent its meaning.
2. **Decoder** generates the target sentence **one word at a time**.
   - It starts with the special `<sos>` token (start of sentence).
   - At each step, it predicts the next token.
   - The prediction (or sometimes the true token) is fed back in as the next input.
3. The model’s predictions are compared to the correct target tokens using a **cross-entropy loss** function.
4. The gradients are computed and used to update the model weights.

---

### 🎓 Teacher Forcing

During training, we don’t always feed the decoder its own predictions — that would make learning very slow in the early stages.  
Instead, with **teacher forcing**, we sometimes feed it the **correct token** from the target sentence.

- With probability `p = 0.7`, the decoder gets the **true token** from the dataset.  
- With probability `1 - p`, it uses its **own previous prediction**.  

This mix helps the model learn faster and prevents it from drifting too far when it makes mistakes.

---

### ⚡ Mixed Precision Training (AMP)

When running on a GPU, we can use **Automatic Mixed Precision (AMP)** to speed up training.  
AMP uses 16-bit floating point numbers where possible — reducing memory usage and increasing efficiency —  
while keeping 32-bit precision for important operations like gradient updates.

---

### 🔍 Evaluation

To check how well the model performs:

- The **validation loop** runs the model on held-out data.
- Validation loss uses gold previous tokens (teacher-forced likelihood), while exact-match accuracy uses greedy autoregressive decoding. These measure different behaviors.
- We compute:
  - **Validation loss** — how close predictions are to the true tokens.
  - **Exact match accuracy** — how often the entire output sentence matches the ground truth.

---

### 🛑 Early Stopping

To prevent overfitting, we use **early stopping**:
- After each epoch, if validation loss stops improving for a few rounds, we stop training.
- The best-performing model (lowest validation loss) is automatically restored.

---

### 📈 Plot for Loss Curves

After training finishes, we plot the **training and validation loss curves** to see how the model improved over time.

---

> 💡 **Summary:**  
> The training loop alternates between encoding, decoding, computing loss, and updating model weights —  
> using teacher forcing to guide learning and early stopping to ensure good generalization.


In [ ]:
# Automatic mixed precision is useful on CUDA and safely disabled on CPU.
use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)


def decoder_step(model, inp, enc_out, hidden, key_mask):
    if hasattr(model.dec, "attn") or hasattr(model.dec, "mh"):
        return model.dec(inp, enc_out, hidden, key_mask=key_mask)
    logits, hidden = model.dec(inp, hidden)
    return logits, hidden, None


def train_one_epoch(model, loader, optimizer, criterion, teacher_forcing=0.7):
    model.train()
    total = 0.0

    for src, tgt in loader:
        optimizer.zero_grad(set_to_none=True)
        key_mask = src.ne(PAD_SRC)

        with torch.autocast(
            device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp
        ):
            enc_out, hidden = model.enc(src)
            inp = tgt[:, :1]
            logits_steps = []

            for t in range(1, tgt.size(1)):
                step_logits, hidden, _ = decoder_step(
                    model, inp, enc_out, hidden, key_mask
                )
                logits_steps.append(step_logits)
                use_gold = random.random() < teacher_forcing
                inp = tgt[:, t : t + 1] if use_gold else step_logits.argmax(-1)

            logits = torch.cat(logits_steps, dim=1)
            loss = criterion(
                logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1)
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()

    return total / len(loader)


@torch.no_grad()
def decode_greedy(model, src, max_len):
    model.eval()
    enc_out, hidden = model.enc(src)
    inp = torch.full(
        (src.size(0), 1),
        tgt_stoi["<sos>"],
        device=src.device,
        dtype=torch.long,
    )
    key_mask = src.ne(PAD_SRC)
    outputs = []

    for _ in range(max_len - 1):
        step_logits, hidden, _ = decoder_step(model, inp, enc_out, hidden, key_mask)
        inp = step_logits.argmax(-1)
        outputs.append(inp)

    return torch.cat(outputs, dim=1)


@torch.no_grad()
def evaluate(model, loader, criterion, max_len):
    model.eval()
    total_loss = 0.0
    exact = 0
    n = 0

    for src, tgt in loader:
        enc_out, hidden = model.enc(src)
        inp = tgt[:, :1]
        steps = []
        key_mask = src.ne(PAD_SRC)

        # Gold previous tokens give a stable teacher-forced validation loss.
        for t in range(1, tgt.size(1)):
            step_logits, hidden, _ = decoder_step(
                model, inp, enc_out, hidden, key_mask
            )
            steps.append(step_logits)
            inp = tgt[:, t : t + 1]

        logits = torch.cat(steps, dim=1)
        loss = criterion(
            logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1)
        )
        total_loss += loss.item()

        pred = decode_greedy(model, src, max_len)
        for i in range(src.size(0)):
            gold = tgt[i, 1:]
            eos_positions = gold.eq(tgt_stoi["<eos>"]).nonzero(as_tuple=True)[0]
            length = eos_positions[0].item() + 1 if len(eos_positions) else gold.size(0)
            exact += pred[i, :length].eq(gold[:length]).all().item()
            n += 1

    return total_loss / len(loader), exact / max(n, 1)


In [ ]:
# Early stopping to avoid overfitting
class EarlyStopper:
    def __init__(self, patience=2, mode="min"):
        self.patience = patience  # How many epochs to wait before stopping
        self.mode = mode          # "min" for loss, "max" for accuracy
        self.best = float("inf") if mode == "min" else -float("inf")
        self.wait = 0             # How many epochs since last improvement
        self.state = None         # Save best model parameters
    
    def step(self, metric, model):
        # Check if performance improved
        improved = (metric < self.best) if self.mode == "min" else (metric > self.best)
        if improved:
            self.best = metric
            self.wait = 0
            # Save model weights to restore later
            self.state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.wait += 1
        return improved
    
    def should_stop(self):
        # Stop training if patience exceeded
        return self.wait >= self.patience
    
    def load_best(self, model):
        # Restore best weights
        if self.state:
            model.load_state_dict(self.state, strict=True)


# Main training loop
def train_model(model, name):
    print(f"\n=== Training {name} ===  batches/epoch={len(train_loader)}")
    
    # Optimizer, loss, and early stopper
    opt = optim.Adam(model.parameters(), lr=CFG.lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(ignore_index=PAD_TGT)
    stop = EarlyStopper(patience=2, mode="min")
    
    tr_hist, vl_hist, acc_hist = [], [], []
    
    # Loop over epochs
    for ep in range(CFG.epochs):
        t0 = time.time()
        
        # Training and validation
        tr = train_one_epoch(model, train_loader, opt, crit, CFG.teacher_forcing)
        vl, acc = evaluate(model, val_loader, crit, CFG.max_len)
        
        tr_hist.append(tr)
        vl_hist.append(vl)
        acc_hist.append(acc)
        
        print(f"Epoch {ep+1}/{CFG.epochs} | Train {tr:.3f} | Val {vl:.3f} | Acc {acc:.2f} | {time.time()-t0:.1f}s")
        
        # Check early stopping
        stop.step(vl, model)
        if stop.should_stop():
            print("Early stopping.")
            break
    
    # Load best-performing model weights
    stop.load_best(model)
    
    # Plot training and validation losses
    plt.figure()
    plt.plot(tr_hist, label="Train")
    plt.plot(vl_hist, label="Val")
    plt.legend()
    plt.title(name)
    plt.show()
    
    return model

## 👁️ Visualizing Attention Weights

When we add **attention** to a Seq2Seq model, the decoder doesn’t rely only on a single context vector anymore.  
Instead, at each decoding step, it **computes a weighted combination of the encoder’s outputs**, deciding which parts of the input sentence are most relevant.

These weights are called **attention scores** — they tell us *where the model is "looking"* when predicting each target word.

---

### 🎨 What This Visualization Shows

This script extracts the attention weights from a trained model and plots them as a **heatmap**:

- **Rows** represent the words in the **source sentence** (input).  
- **Columns** represent the words in the **target sentence** (output).  
- The **color intensity** shows the attention weight assigned to each source position for a generated target token.

If the model uses **multi-head attention**, we visualize:
1. The attention map for **each head** separately.  
2. The **average attention map** across all heads.

Different heads may show different patterns, but a visual pattern alone does not prove that a head represents a particular linguistic concept.

---

> 💡 **Tip:**  
> A diagonal pattern can be consistent with word alignment  
> (e.g., “dog → Hund”, “runs → läuft”).


> ⚠️ **Interpretation note:** attention weights are useful model diagnostics, but they are not by themselves causal explanations of a prediction.


In [ ]:
@torch.no_grad()
def one_example_attention(model, pair_idx=0, title="Attention"):
    src, _ = val_pairs[pair_idx]
    src_batch = src.unsqueeze(0).to(DEVICE)
    enc_out, hidden = model.enc(src_batch)
    key_mask = src_batch.ne(PAD_SRC)

    inp = torch.full(
        (1, 1), tgt_stoi["<sos>"], device=DEVICE, dtype=torch.long
    )
    attn_steps = []
    predicted_ids = []

    for _ in range(CFG.max_len - 1):
        logits, hidden, attn = decoder_step(
            model, inp, enc_out, hidden, key_mask
        )
        if attn is None:
            print("No attention to visualize.")
            return

        if hasattr(model.dec, "mh"):
            # (B * heads, 1, source_len) -> (heads, 1, source_len), B=1
            attn = attn.view(model.dec.mh.h, 1, 1, src.size(0)).squeeze(1)
        attn_steps.append(attn)

        inp = logits.argmax(-1)
        token_id = inp.item()
        predicted_ids.append(token_id)
        if token_id == tgt_stoi["<eos>"]:
            break

    src_len = src.ne(PAD_SRC).sum().item()
    src_tokens = [src_itos[index.item()] for index in src[:src_len]]
    tgt_tokens = [tgt_itos[index] for index in predicted_ids]

    if hasattr(model.dec, "mh"):
        attention = torch.cat(attn_steps, dim=1).detach().cpu().numpy()
    else:
        attention = torch.cat(attn_steps, dim=1)[0].detach().cpu().numpy()[None, ...]

    attention = attention[:, : len(tgt_tokens), :src_len]
    n_heads = attention.shape[0]

    if n_heads > 1:
        cols = 2
        rows = math.ceil(n_heads / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 3 * rows), squeeze=False)
        vmax = attention.max()
        for head, ax in enumerate(axes.flat):
            if head >= n_heads:
                ax.axis("off")
                continue
            sns.heatmap(
                attention[head].T,
                xticklabels=tgt_tokens,
                yticklabels=src_tokens,
                cmap="viridis",
                vmin=0.0,
                vmax=vmax,
                ax=ax,
            )
            ax.set_title(f"Head {head + 1}")
            ax.set_xlabel("Generated target tokens")
            ax.set_ylabel("Source tokens")
            ax.tick_params(axis="x", rotation=45)
        fig.suptitle(title + " (per head)")
        fig.tight_layout()
        plt.show()

    mean_attention = attention.mean(axis=0)
    plt.figure(figsize=(max(7, 1.0 * len(tgt_tokens)), max(4, 0.55 * len(src_tokens))))
    sns.heatmap(
        mean_attention.T,
        xticklabels=tgt_tokens,
        yticklabels=src_tokens,
        cmap="viridis",
    )
    plt.title(title + (" (mean across heads)" if n_heads > 1 else ""))
    plt.xlabel("Generated target tokens")
    plt.ylabel("Source tokens")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


## 🚀 Training and Visualizing

Now it’s time to bring everything together!  
We’ll **train our Seq2Seq models** and then **visualize their behavior** to see how well they learned to translate.

### 🧠 What We'll Do
1. **Train the models** (with and without attention) on our English–German dataset.  
   - The model will learn to generate German translations word by word.  
   - We'll track training and validation loss to monitor learning progress.  
2. **Evaluate the models** on examples to see how accurate the translations are.  
3. **Visualize attention maps** to understand *where the model is "looking"* when generating each target word.

### 🎯 Why This Matters
Visualization helps us interpret what the model learned.  
- The **no-attention model** relies on a single context vector, which can lose information on long sentences.  
- The **attention models** dynamically focus on different source words, offering both better performance and greater interpretability.

Let’s start by selecting an example sentence that we’ll use later for comparison.


In [ ]:
# Choose one example sentence pair for visualization

# Pick the first training example (you can change the index)
example_idx = 2

# Get the source (English) and target (German) tensors
# You can also use an example from the train_pairs
src, tgt = val_pairs[example_idx]

# Convert token IDs back to readable words, skipping <pad> tokens
src_tokens = [src_itos[i.item()] for i in src if i.item() != PAD_SRC]
tgt_tokens = [tgt_itos[i.item()] for i in tgt if i.item() != PAD_TGT]

# Print the selected example for inspection
print(f"Example ID: {example_idx}")
print("Source:", " ".join(src_tokens))
print("Target:", " ".join(tgt_tokens))

In [ ]:
# 1) Seq2Seq (no attention)
model1 = Seq2SeqNoAttn(CFG.emb, CFG.hid, CFG.dropout).to(DEVICE)
train_model(model1, "Seq2Seq (No Attention)")

In [ ]:
# 2) Luong
model2 = Seq2SeqLuong(CFG.emb, CFG.hid, CFG.dropout).to(DEVICE)
train_model(model2, "Luong Attention")

In [ ]:
# 2) Luong Visualization
one_example_attention(model2, pair_idx=example_idx, title="Luong Attention")

In [ ]:
# 3) Scaled Dot-Product (1 head)
model3 = Seq2SeqMH(CFG.emb, CFG.hid, 1, CFG.dropout).to(DEVICE)
train_model(model3, "Scaled Dot-Product (QKV, 1 head)")

In [ ]:
# 3) Scaled Dot-Product (1 head) Visualization
one_example_attention(model3, pair_idx=example_idx, title="Scaled Dot-Product (1 head)")

In [ ]:
# 4) Multi-Head (4 heads)
model4 = Seq2SeqMH(CFG.emb, CFG.hid, heads, CFG.dropout).to(DEVICE)
train_model(model4, "Multi-Head Attention (4 heads)")

In [ ]:
# 4) Multi-Head (4 heads) Visualization
one_example_attention(model4, pair_idx=example_idx, title="Multi-Head Attention (mean over heads)")

In [ ]:
# 5) Self-Attention Encoder + MH decoder
model5 = Seq2SeqSelfEnc(CFG.emb, CFG.hid, heads, CFG.dropout).to(DEVICE)
train_model(model5, "Self-Attention Encoder + MH Decoder");

In [ ]:
# 5) Self-Attention Encoder + MH decoder Visualization
one_example_attention(model5, pair_idx=example_idx, title="Self-Attention Encoder (decoder cross-attn)")


## 🧩 Summary: Comparing Different Attention Mechanisms

Now that we’ve implemented several attention-based models, let’s review how they differ and what each one contributes to the Seq2Seq architecture.

| **Type** | **Where It’s Used** | **Core Idea** | **Key Strengths** | **Limitations** |
|:----------|:--------------------|:---------------|:------------------|:----------------|
| **Luong (Dot-Product) Attention** | Between encoder and decoder (cross-attention) | Computes a dot product between the decoder’s hidden state (query) and encoder outputs (keys). Uses a single attention distribution. | Simple and efficient; helps the decoder focus on relevant input words. | Only one attention “view” of the input — may miss complex relationships. |
| **Scaled Dot-Product (Multi-Head) Attention** | Between encoder and decoder (cross-attention) | Uses multiple sets of queries, keys, and values in parallel (“heads”). Different heads can learn complementary attention patterns. | Richer contextual understanding; can capture multiple linguistic patterns at once. | More computationally expensive; needs careful scaling. |
| **Self-Attention (Encoder)** | Within the encoder itself | Each token attends to *all other tokens* in the same sequence (Q = K = V = input embeddings). | Captures long-range dependencies and global context efficiently; parallelizable. | Loses the sequential inductive bias of RNNs (order must be learned). |

---

### 💡 Key Takeaways

- **Luong attention** helps the decoder focus on the right source words.  
- **Multi-head attention** lets the model focus on multiple aspects of the sequence at once.  
- **Self-attention** allows the encoder to understand context across the entire input without recurrence.

Together, these mechanisms form the foundation of modern architectures like the **Transformer**, which replaces recurrence entirely with layers of multi-head self-attention.


## 🧠 Visualizing Attention in a Pretrained Transformer (T5)

So far, we’ve implemented and visualized different types of attention mechanisms ourselves — from simple Luong attention to multi-head and self-attention encoders.

Now, let’s look at how **modern Transformer models** like **T5 (Text-to-Text Transfer Transformer)** use these attention mechanisms *at scale*.

T5 uses:
- **Self-attention** in both the encoder and decoder.  
- **Cross-attention** between the decoder and encoder (just like in our Seq2Seq models).  
- **Multiple layers and heads**, each capturing different linguistic patterns.

By visualizing T5’s attention maps, we can see how a pretrained Transformer distributes attention across words when performing translation or text generation tasks.

`sentencepiece` was included in the setup cell because T5 uses a SentencePiece tokenizer. If the package was installed only now, restart the kernel before continuing.


## 🌐 Downloading the public T5 checkpoint

The checkpoint below is public, so students do **not** need a Hugging Face account or access token. The first run needs internet access and downloads model files into the user's cache. On a managed JupyterHub, the instructor can pre-cache the model if compute nodes do not have outbound internet access.


In [ ]:
# No login is required for the public google-t5/t5-small checkpoint.


## 🧩 Visualizing Cross-Attention in a Pretrained Transformer (T5)

To conclude our exploration of attention, let’s look inside a **real Transformer model** — **T5 (Text-to-Text Transfer Transformer)**.  
Unlike our earlier Seq2Seq models, T5 has **attention everywhere**:
- **Self-attention** in the encoder (each input word attends to all others).  
- **Cross-attention** in the decoder (each generated word attends to relevant source words).  
- **Self-attention** again in the decoder (for coherence in generation).

In this final step, we’ll visualize **cross-attention** from the *last decoder layer* of **T5-base**,  
showing how it aligns **English source tokens** with **German target tokens** during translation.

---

### 💡 What You’ll See
- **Bright regions** on the heatmap indicate strong attention between specific word pairs.  
- **Rows** correspond to source tokens (English).  
- **Columns** correspond to generated target tokens (German).  
- We’ll visualize both:
  - The **average across all heads** (overall alignment).  
  - The **individual attention heads** (different linguistic focuses).

This gives a real look into how Transformer attention mechanisms power modern translation models.


In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

T5_DEVICE = DEVICE
model_name = "google-t5/t5-small"


In [ ]:
# `output_attentions` is requested during the forward passes below.
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Current Transformers versions default to an optimized SDPA backend that does
# not expose attention weights. Eager attention is required for visualization.
model = T5ForConditionalGeneration.from_pretrained(
    model_name, attn_implementation="eager"
).to(T5_DEVICE)
model.eval()


In [ ]:
max_len = 64  # limit decoder steps to avoid runaway generation
eos_token_id = tokenizer.eos_token_id
if eos_token_id is None:
    eos_token_id = tokenizer.convert_tokens_to_ids("</s>")

# Load an example from our dataset
example_idx = 2
src, tgt = val_pairs[example_idx]

# Convert token IDs to words and remove special tokens
src_tokens = [t for t in [src_itos[i.item()] for i in src] if t not in ["<pad>", "<sos>", "<eos>"]]
tgt_tokens_true = [t for t in [tgt_itos[i.item()] for i in tgt] if t not in ["<pad>", "<sos>", "<eos>"]]

src_sentence = " ".join(src_tokens)
tgt_sentence = " ".join(tgt_tokens_true)

print(f"Example ID: {example_idx}")
print(f"Source (dataset): {src_sentence}")
print(f"Target (reference): {tgt_sentence}")

In [ ]:
# Prepare input
task_prefix = "translate English to German: "
input_text = task_prefix + src_sentence
inputs = tokenizer([input_text], return_tensors="pt").to(T5_DEVICE)

# Encode source once
encoder_outputs = model.encoder(**inputs)

# Decoder starts with <pad> (T5 uses <pad> as decoder_start_token)
decoder_input_ids = torch.tensor([[model.config.decoder_start_token_id]], device=T5_DEVICE)

# To collect
decoded_token_ids = []
cross_attn_weights = []

for step in range(max_len):
    outputs = model(
        encoder_outputs=encoder_outputs,
        decoder_input_ids=decoder_input_ids,
        attention_mask=inputs["attention_mask"],
        output_attentions=True,
        return_dict=True,
        use_cache=False
    )

    # Get next token
    next_token_logits = outputs.logits[:, -1, :]
    next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)

    # Get cross-attention from last decoder layer
    cross_attn = outputs.cross_attentions[-1]

    # Handle shape: (n_heads, tgt_len, src_len) or (1, n_heads, tgt_len, src_len)
    if cross_attn.dim() == 4:
        cross_attn = cross_attn[0]

    # Store attention for current token
    cross_attn_weights.append(cross_attn[:, -1, :])  # (n_heads, src_len)

    # Append token and update input
    decoded_token_ids.append(next_token_id.item())
    decoder_input_ids = torch.cat([decoder_input_ids, next_token_id], dim=1)

    if next_token_id.item() == eos_token_id:
        break


# Decode tokens
tgt_tokens_mt = tokenizer.convert_ids_to_tokens(decoded_token_ids)
src_tokens_mt = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

# Remove prefix (and leading pad) so visualization reflects only the source sentence
input_ids_full = inputs["input_ids"][0].tolist()
prefix_ids = tokenizer(task_prefix, add_special_tokens=False).input_ids
pad_token_id = tokenizer.pad_token_id
prefix_offset = 0

if input_ids_full and input_ids_full[0] == pad_token_id:
    prefix_offset = 1
    ids_wo_pad = input_ids_full[1:]
else:
    ids_wo_pad = input_ids_full

if prefix_ids and ids_wo_pad[: len(prefix_ids)] == prefix_ids:
    prefix_offset += len(prefix_ids)

if prefix_offset:
    src_tokens_mt = src_tokens_mt[prefix_offset:]

print(tgt_tokens_mt)
print(src_tokens_mt)

In [ ]:
# Keep SentencePiece's `▁` word-boundary marker visible; stripping it can
# create empty heatmap labels when the marker is emitted as its own token.
def clean_tokens(tokens):
    specials = {"<pad>", "</s>", "<unk>"}
    return [token for token in tokens if token not in specials]


src_tokens_clean = clean_tokens(src_tokens_mt)
tgt_tokens_clean = clean_tokens(tgt_tokens_mt)
print(tgt_tokens_clean)
print(src_tokens_clean)

# Stack attention weights: (target_len, heads, source_len) ->
# (heads, target_len, source_len)
A = torch.stack(cross_attn_weights).detach().cpu().numpy().transpose(1, 0, 2)

if prefix_offset:
    A = A[:, :, prefix_offset:]

# The only filtered special token is the final </s>, so prefix slicing followed
# by length slicing keeps labels and attention columns aligned.
src_len = len(src_tokens_clean)
tgt_len = len(tgt_tokens_clean)
A = A[:, :tgt_len, :src_len]


In [ ]:
# Average across heads → (src_len, tgt_len)
A_mean = A.mean(axis=0).T  # Transpose for heatmap

# Plot mean cross-attention
plt.figure(figsize=(1.2 * tgt_len, 0.8 * src_len))
sns.heatmap(
    A_mean,
    xticklabels=tgt_tokens_clean,
    yticklabels=src_tokens_clean,
    cmap="viridis",
    cbar=True,
    cbar_kws={"shrink": 0.6}
)
plt.title("T5 Cross-Attention (Mean Across Heads)", pad=20)
plt.xlabel("German (Target Tokens)")
plt.ylabel("English (Source Tokens)")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
ax = plt.gca()
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize Individual Attention Heads
show_per_head = True
if show_per_head:
    n_heads = A.shape[0]
    cols, rows = 2, int(np.ceil(n_heads / 2))
    fig_w = 1.2 * tgt_len * cols
    fig_h = 0.9 * src_len * rows

    plt.figure(figsize=(fig_w, fig_h))
    for h in range(n_heads):
        plt.subplot(rows, cols, h + 1)
        sns.heatmap(
            A[h].T,
            xticklabels=tgt_tokens_clean,
            yticklabels=src_tokens_clean,
            cmap="viridis",
            cbar=True,
            cbar_kws={"shrink": 0.6, "pad": 0.03}
        )
        plt.title(f"Head {h+1}")
        plt.xticks(rotation=45, ha="right")
        plt.yticks(rotation=0)
        ax = plt.gca()
        ax.xaxis.tick_top()
        ax.xaxis.set_label_position('top')
    plt.suptitle("T5 Cross-Attention (Last Decoder Layer, Per Head)", y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()

## 🧭 Reflection: What We Learned About Attention

Over the course of this notebook, you built and explored several neural sequence models - progressing from a simple encoder–decoder to powerful attention-based architectures.

---

### 🧩 Key Takeaways

1. **Seq2Seq without attention** compresses the entire input into one context vector.  
   - Works for short sentences, but struggles with long-term dependencies.  

2. **Luong (dot-product) attention** lets the decoder selectively focus on relevant source words.  
   - The model learns to “align” input and output words dynamically.  

3. **Multi-head attention** expands this idea by learning multiple focus patterns at once.  
   - Different heads can learn complementary patterns, although a head's linguistic role should not be assumed from a heatmap alone.  

4. **Self-attention** lets an encoder process tokens without recurrence, but it needs positional information to represent word order.  
   - Every token directly interacts with all others, enabling efficient parallel computation.  

5. **Transformers (like T5)** combine these ideas into deep, scalable architectures  
   that dominate modern NLP tasks such as translation, summarization, and question answering.

---

## 🧠 Your Exercise: Experiment, Extend, and Evaluate

Here are some ideas to deepen your understanding and creativity:

### 🧪 1. Experiment with the Data
- Replace the current dataset with another parallel corpus (e.g., English–French or English–Spanish).  
- Or, create a **mini dataset** from your own bilingual text.  
- Observe how the model handles a different language pair.
- You can generate sentence pairs using this website: https://tatoeba.org/en/downloads

### ⚙️ 2. Tune Hyperparameters
- Change embedding size, hidden size, dropout rate, or number of attention heads.  
- Plot how these changes affect training/validation loss.  
- Try **reducing** or **increasing** the teacher forcing ratio — how does it influence convergence?

### 🧮 3. Add a Proper Test Set
- Currently, we used only training and validation splits.  
- Create a **held-out test set** (e.g., 10% of your data) to evaluate true generalization performance.  
- Run inference on this unseen data and compare results.

### 📏 4. Evaluate Model Outputs Quantitatively
- Implement **BLEU score** evaluation using libraries like `nltk`.
- Evaluate model performance on your test set.
- Save the trained model for future inference.